# LFY033 – Introduktion till programmering i Python och artificiell intelligens

## Föreläsning 4: Dataanalys med pandas och visualisering


# Förra gången

Vi arbetade med:

- funktioner
- dictionaries
- att strukturera kod och data

Nu byter vi fokus: **från att skriva program till att använda Python för att undersöka data.**


# Arbetsflödet idag

**Läs data → inspektera → välj/filtrera → beräkna → sammanfatta → visualisera → tolka**

Det här arbetsflödet kommer tillbaka även när vi börjar med maskininlärning.


# Del 1: Data som vårt eget program skapar

Vi börjar med något bekant: en simulering av kaströrelse.

Funktionen ger oss två listor som hör ihop: `x_positions` och `y_positions`.


# Kaströrelse

Ni behöver inte fokusera på fysiken här. Titta i stället på **formen på datan som funktionen producerar**.


In [ ]:
import math
import numpy as np


def kaströrelse(initial_hastighet, vinkel, tid, antal_steg):
    vinkel_radianer = math.radians(vinkel)
    vx = initial_hastighet * math.cos(vinkel_radianer)
    vy = initial_hastighet * math.sin(vinkel_radianer)

    tider = np.linspace(0, tid, antal_steg)
    x_positions = []
    y_positions = []

    for t in tider:
        x = vx * t
        y = vy * t - 0.5 * 9.81 * t**2

        if y < 0:
            break

        x_positions.append(x)
        y_positions.append(y)

    return x_positions, y_positions


# Kör simuleringen

**Vad tror ni att de två variablerna innehåller?**


In [ ]:
x_positions, y_positions = kaströrelse(
    initial_hastighet=20,
    vinkel=45,
    tid=10,
    antal_steg=200
)


In [ ]:
print(x_positions[:5])
print(y_positions[:5])


# Två listor som hör ihop

`x_positions[10]` och `y_positions[10]` beskriver samma punkt i rörelsen.

Det fungerar – men tabulär data blir snart besvärlig om vi måste hålla reda på många parallella listor.


# pandas

`pandas` är ett bibliotek för tabulär data.

Vi importerar det nästan alltid med namnet `pd`.


In [ ]:
import pandas as pd


# Från listor till en DataFrame

Kolumnnamnen berättar vad värdena betyder.


In [ ]:
trajectory = pd.DataFrame({
    "x": x_positions,
    "y": y_positions
})

trajectory.head()


# Några snabba sätt att inspektera tabellen

Vi behöver oftast inte skriva ut allt.


In [ ]:
print(trajectory.shape)
print(trajectory.columns)


# Välj en kolumn

Vad tror ni att detta ger?


In [ ]:
trajectory["y"].head()


# Filtrera rader

Först skapar vi ett villkor.


In [ ]:
trajectory["y"] > 10


# Sedan använder vi villkoret som filter


In [ ]:
trajectory[trajectory["y"] > 10].head()


# Samma idé som `for` + `if`

Tidigare hade vi kunnat loopa över alla höjder och spara de som var över 10.

Med pandas uttrycker vi frågan direkt på hela kolumnen.


# Hur högt når projektilen?


In [ ]:
trajectory["y"].max()


# Var inträffar maxhöjden?


In [ ]:
max_height = trajectory["y"].max()
trajectory[trajectory["y"] == max_height]


# Ny kolumn

Beräkningar på en kolumn görs rad för rad utan att vi själva skriver en loop.


In [ ]:
trajectory["distance_from_start"] = (
    trajectory["x"]**2 + trajectory["y"]**2
) ** 0.5

trajectory.head()


# Från tabell till bild

Vad tror ni att formen blir om vi ritar `y` mot `x`?


In [ ]:
trajectory.plot(x="x", y="y")


# Varför visualisera?

Tabellen innehåller samma information som diagrammet.

Men i diagrammet ser vi **rörelsens form** direkt.

Visualisering är ett analysverktyg – inte bara dekoration.


# Del 2: Data från en fil

Nu gör vi samma typ av arbete med data som någon annan har samlat in.

Vi återanvänder CO₂-datan från laboration 3.


# Läs in CSV-data

`read_csv()` skapar en DataFrame direkt från filen.


In [ ]:
co2data = pd.read_csv("co2-mm-mlo-0-2014.csv")
co2data.head()


# Börja alltid med att undersöka datan


In [ ]:
print(co2data.shape)
print(co2data.columns)


# Begränsa analysen

Vi arbetar vidare med år 1900 och framåt.


In [ ]:
co2 = co2data[co2data["year"] >= 1900].copy()
co2.head()


# Välj bara de kolumner vi är intresserade av


In [ ]:
co2[[
    "year",
    "data_mean_global",
    "data_mean_nh",
    "data_mean_sh"
]].head()


# Filtrera med flera villkor

Vilka rader uppfyller **båda** villkoren?


In [ ]:
co2[
    (co2["year"] >= 1950) &
    (co2["data_mean_global"] < 320)
]


# `&` i stället för `and`

När vi jämför hela pandas-kolumner använder vi:

- `&` för **och**
- `|` för **eller**

Sätt parenteser runt varje jämförelse.


# Skapa något som inte fanns i originaldatan

Hur stor är skillnaden mellan norra och södra halvklotet?


In [ ]:
co2["nh_minus_sh"] = (
    co2["data_mean_nh"] - co2["data_mean_sh"]
)

co2[["year", "nh_minus_sh"]].head()


# Sammanfatta en kolumn


In [ ]:
print("min:", co2["data_mean_global"].min())
print("max:", co2["data_mean_global"].max())
print("medel:", co2["data_mean_global"].mean())


# Paus


# Sammanfatta grupper

Anta att vi vill jämföra **årtionden** i stället för enskilda år.

Först skapar vi en kolumn som anger årtionde.


In [ ]:
co2["decade"] = (co2["year"] // 10) * 10
co2[["year", "decade"]].head(12)


# `groupby`

**Dela upp raderna i grupper → beräkna något för varje grupp.**


In [ ]:
decade_means = co2.groupby("decade")["data_mean_global"].mean()
decade_means


# Sortera data

Vilka år hade högst global CO₂-halt?


In [ ]:
co2.sort_values("data_mean_global", ascending=False).head()


# Nu visualiserar vi tidsserien


In [ ]:
co2.plot(
    x="year",
    y="data_mean_global",
    title="Global CO₂-koncentration över tid"
)


# Vad ser vi?

Fundera på två olika frågor:

1. Vad kan vi läsa ut **direkt** ur diagrammet?
2. Vad kan vi **inte** dra för slutsats bara från diagrammet?


# Flera serier i samma diagram

Det här passar när vi vill jämföra utvecklingen över samma x-axel.


In [ ]:
co2.plot(
    x="year",
    y=["data_mean_nh", "data_mean_sh"],
    title="CO₂ på norra och södra halvklotet"
)


# Linjediagram

Bra när x-axeln har en naturlig ordning – särskilt **tid**.

Men alla data ska inte kopplas ihop med en linje.


# Scatterplot

Bra när vi vill undersöka relationen mellan **två numeriska variabler**.


In [ ]:
co2.plot.scatter(
    x="data_mean_nh",
    y="data_mean_sh",
    title="Norra mot södra halvklotet"
)


# Vad verkar scatterplotten visa?

Punkterna verkar ligga nära ett mönster.

Nästa steg i kursen blir att fråga:

> Kan vi lära en modell ett sådant samband från data?


# Histogram

Ett histogram visar hur värdena i **en variabel** är fördelade.


In [ ]:
co2["nh_minus_sh"].hist()


# Gör diagrammet begripligt

Rubrik, axelrubriker och enheter hjälper läsaren förstå vad bilden betyder.


In [ ]:
import matplotlib.pyplot as plt

co2.plot(x="year", y="data_mean_global", legend=False)
plt.xlabel("År")
plt.ylabel("CO₂ (ppm)")
plt.title("Global atmosfärisk CO₂")
plt.show()


# Saknade värden – en snabb kontroll

Verkliga dataset är ofta ofullständiga.


In [ ]:
co2data.isna().sum()


# Ingen automatisk regel

Saknade värden kan exempelvis:

- tas bort
- ersättas
- lämnas kvar

Vilket som är rätt beror på **vad värdet betyder och vilken fråga vi ställer**.


# Tre diagramtyper räcker långt

### Linje
Utveckling över en ordnad x-axel, ofta tid.

### Scatter
Relation mellan två numeriska variabler.

### Histogram
Fördelningen av en numerisk variabel.


# Arbetsflödet igen

**Läs → inspektera → filtrera → beräkna → sammanfatta → visualisera → tolka**

Pandas hjälper oss med hela kedjan.


# Nästa gång: maskininlärning

Hittills har vi främst frågat:

> **Vad berättar datan?**

Nästa gång frågar vi:

> **Kan data användas för att förutsäga ett värde vi inte känner till?**


# Efter idag ska ni kunna

- läsa in CSV-data i en DataFrame
- inspektera, välja och filtrera data
- skapa nya kolumner
- sammanfatta data med enkla aggregat och `groupby`
- göra linjediagram, scatterplots och histogram
- tolka vad ett diagram visar – och inte visar


# Laboration 4

I labben fortsätter ni med samma CO₂-data.

Den här gången ska ni inte bara bearbeta tabellen utan också **göra egna diagram och använda dem för att besvara frågor om datan**.
